In [26]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

a = 2 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'b' # unit of length
na = 10  # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom = atoms,
            basis = 'sto6g',
            verbose = 4,
            unit = 'B',
            symmetry = 0,
            charge = 0,
            spin = 0,
            max_memory = 20000,
            )

mf = scf.RHF(mol).density_fit()
mf.kernel()

frozen = 0
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-23-generic', version='#23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed May  6 14:20:25 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 10
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      un

In [27]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.11505014
CCSD Corr = -0.18568664
CCSD(T) Corr = -0.18796563


In [28]:
nocc = np.count_nonzero(mf.mo_occ)
mo = mf.mo_coeff
mo_occ = mf.mo_coeff[:,:nocc]
mo_vir = mf.mo_coeff[:,nocc:]
mo_e = mf.mo_energy
print(mo_occ.shape)
print(mo_vir.shape)
print(mo_e)

(10, 5)
(10, 5)
[-0.6601326  -0.61116877 -0.5300571  -0.41558466 -0.27230882  0.14077227
  0.35171329  0.5902159   0.82955314  1.02015204]


In [29]:
fock_ao = mf.get_fock()
print(fock_ao)

[[-2.29529560e-01 -4.54007737e-01 -1.35211102e-01 -7.32738499e-04
   1.70249716e-03 -7.73667844e-03 -1.34498049e-03  3.64663758e-03
   5.41513828e-04 -2.18072827e-03]
 [-4.54007737e-01 -3.23365296e-01 -3.77932609e-01 -1.33246788e-01
  -1.88940659e-02  1.20386331e-03 -7.16148146e-04 -1.21068051e-03
   1.26781025e-04  5.41513828e-04]
 [-1.35211102e-01 -3.77932609e-01 -3.28580834e-01 -4.54973254e-01
  -1.34073155e-01 -3.03689686e-03  1.27695017e-03 -7.05742257e-03
  -1.21068051e-03  3.64663758e-03]
 [-7.32738499e-04 -1.33246788e-01 -4.54973254e-01 -3.30085464e-01
  -3.83971583e-01 -1.33789358e-01 -1.83862867e-02  1.27695017e-03
  -7.16148146e-04 -1.34498049e-03]
 [ 1.70249716e-03 -1.88940659e-02 -1.34073155e-01 -3.83971583e-01
  -3.30274607e-01 -4.53464886e-01 -1.33789358e-01 -3.03689686e-03
   1.20386331e-03 -7.73667844e-03]
 [-7.73667844e-03  1.20386331e-03 -3.03689686e-03 -1.33789358e-01
  -4.53464886e-01 -3.30274607e-01 -3.83971583e-01 -1.34073155e-01
  -1.88940659e-02  1.70249716e-03

In [33]:
def get_fock_ov(fock, nocc):
    '''separate the fock into oo, ov, vv blocks'''
    oo = fock[:nocc,:nocc]
    ov = fock[:nocc,nocc:]
    vv = fock[nocc:,nocc:]
    return oo, ov, vv

def larg_offd(mat):
    off_diag = mat - np.diag(np.diag(mat))
    max_val = np.max(np.abs(off_diag))
    return max_val

In [36]:
fock_mo = mo.T @ fock_ao @ mo
foo, fov, fvv = get_fock_ov(fock_mo, nocc)
print(larg_offd(foo))
print(np.max(np.abs(fov)))
print(larg_offd(fvv))

3.7816209226548825e-09
5.254298749562958e-08
5.948618417295215e-09


In [35]:
print(foo)

[[-6.60132600e-01 -6.41847686e-16  3.78162090e-09  3.26128013e-16
   1.47384165e-09]
 [-6.76542156e-16 -6.11168767e-01 -1.16573418e-15 -1.87649793e-09
  -7.63278329e-17]
 [ 3.78162092e-09 -1.13797860e-15 -5.30057092e-01  8.04911693e-16
  -1.86941929e-09]
 [ 3.40005801e-16 -1.87649790e-09  8.32667268e-16 -4.15584659e-01
  -1.77635684e-15]
 [ 1.47384166e-09 -3.81639165e-17 -1.86941929e-09 -1.77982629e-15
  -2.72308863e-01]]


In [39]:
# PM localization
# orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_occ_pm = lo.PipekMezey(mol, mo_occ).kernel()
lo_vir_pm = lo.PipekMezey(mol, mo_vir).kernel()
lo_pm = np.hstack([lo_occ_pm,lo_vir_pm])
dm_lo_pm = mf.make_rdm1(lo_pm, mf.mo_occ)
mf.kernel(dm0=dm_lo_pm)
# while True: # always performing jacobi sweep to avoid trapping in local minimum/saddle point
#     lo_coeff1 = mlo.stability_jacobi()[1]
#     if lo_coeff1 is lo_coeff_pm:
#         break
#     mlo = lo.PipekMezey(mf.mol, lo_coeff1).set(verbose=4)
#     mlo.init_guess = None
#     lo_coeff_pm = mlo.kernel()



******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_stepsize = 0.05
max_iters = 20
kf_interval = 5
kf_trust_region = 5
ah_start_tol = 1000000000.0
ah_start_cycle = 1
ah_level_shift = 0
ah_conv_tol = 1e-12
ah_lindep = 1e-14
ah_max_cycle = 40
ah_trust_region = 3
init_guess = atomic
pop_method = meta_lowdin
Set conv_tol_grad to 0.000316228
macro= 1  f(x)= 2.1539163072738  delta_f= 2.15392  |g|= 0.111571  2 KF 8 Hx
macro= 2  f(x)= 2.1548887745067  delta_f= 0.000972467  |g|= 0.000970992  2 KF 5 Hx
macro= 3  f(x)= 2.154888774507  delta_f= 2.78888e-13  |g|= 8.17796e-06  1 KF 1 Hx
macro X = 3  f(x)= 2.154888774507  |g|= 8.17796e-06  6 intor 5 KF 14 Hx


******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_stepsize = 0.05
max_iters = 20
kf_interval = 5
kf_trust_region = 5
ah_start_tol = 1000000000.0
ah_start_cycle = 1
ah_level_shift = 0
ah_conv_tol = 1e-12
ah_lindep = 1e-14
a

np.float64(-5.206564612945572)

In [40]:
fock_pm = lo_pm.T @ fock_ao @ lo_pm
print(fock_pm)

[[-5.08268567e-01 -1.04966569e-01  3.07239591e-02  3.07238894e-02
  -1.04966674e-01  3.97653915e-14 -1.49139334e-08 -1.49139080e-08
   3.66946423e-08  3.66947209e-08]
 [-1.04966569e-01 -5.07930850e-01 -1.08108438e-01 -1.31725427e-02
   2.89078604e-02 -3.61034155e-08  9.42538862e-09  2.68859525e-08
  -4.03467735e-09 -1.84308170e-08]
 [ 3.07239591e-02 -1.08108438e-01 -4.82561010e-01  7.35385832e-03
  -1.31726901e-02  1.62636219e-08 -5.86557325e-09 -3.69076902e-09
  -2.74736774e-08  1.01982367e-08]
 [ 3.07238894e-02 -1.31725427e-02  7.35385832e-03 -4.82560975e-01
  -1.08108614e-01 -1.62636600e-08 -3.69076343e-09 -5.86558671e-09
   1.01983023e-08 -2.74736020e-08]
 [-1.04966674e-01  2.89078604e-02 -1.31726901e-02 -1.08108614e-01
  -5.07930580e-01  3.61034304e-08  2.68858957e-08  9.42533083e-09
  -1.84308433e-08 -4.03480090e-09]
 [ 3.98127711e-14 -3.61034155e-08  1.62636219e-08 -1.62636601e-08
   3.61034304e-08  6.25554735e-01  2.31424205e-02 -2.31428497e-02
   2.45517607e-01 -2.45516861e-01

In [41]:
foo, fov, fvv = get_fock_ov(fock_pm, nocc)
print(larg_offd(foo))
print(np.max(np.abs(fov)))
print(larg_offd(fvv))

0.10810861378434697
3.669472091070644e-08
0.2473147297388418


In [ ]:
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_coeff_iao = lo.iao.iao(mol, orbocc)
lo_coeff_iao = lo.orth.vec_lowdin(lo_coeff_iao, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)